# Monolingual Semantic Similarity with NLLB-200 Embeddings

NLLB-200 encodes text into a shared multilingual vector space via its encoder.
Sentences with similar meaning map to nearby vectors, regardless of surface form.
Cosine similarity between two such vectors is a reliable proxy for semantic
relatedness within a single language.

**Typical use cases:** duplicate-question detection, sentence clustering,
semantic search, paraphrase mining, retrieval-augmented generation (RAG).

This notebook uses Turkish as the primary language; the same API applies to
all 24 languages supported by TurkicNLP.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import math
import turkicnlp
from turkicnlp import Pipeline

turkicnlp.download("tur", processors=["embeddings"])
embed = Pipeline("tur", processors=["embeddings"])

## 1. Cosine Similarity Between Two Sentences

In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na  = math.sqrt(sum(x**2 for x in a))
    nb  = math.sqrt(sum(y**2 for y in b))
    return dot / (na * nb)

pairs = [
    ("Bugün hava çok güzel.",
     "Dışarısı oldukça sıcak ve güneşli.",
     "high – same topic, different words"),
    ("Ankara Türkiye'nin başkentidir.",
     "Türkiye'nin başkenti Ankara'dır.",
     "very high – near-paraphrase"),
    ("Çocuklar parkta oynuyor.",
     "Ekonomi kriz sürecine girdi.",
     "low – unrelated topics"),
    ("Bu film harikaydı.",
     "Bu film berbattı.",
     "low – opposite sentiment"),
]

print(f"{'Pair':<45} {'Cosine':>7}")
print("-" * 55)
for s1, s2, label in pairs:
    e1 = embed(s1).embedding
    e2 = embed(s2).embedding
    print(f"{label:<45} {cosine(e1, e2):>7.4f}")

## 2. Semantic Ranking — Nearest-Neighbour Search

In [ ]:
# Given a query, rank a corpus by similarity
query  = "Spor salonunda egzersiz yapıyorum."
corpus = [
    "Her sabah koşuya çıkıyorum.",
    "Futbol maçı saat 19:00'da başlıyor.",
    "Yemek pişirmek benim için bir hobi.",
    "Spor yapmak sağlık için çok önemlidir.",
    "Bugün hava yağmurlu.",
    "Fitness merkezi üyeliği aldım.",
    "Ekonomi haberleri hiç iç açıcı değil.",
    "Antrenmanımı tamamladım, çok yorgunum.",
]

q_emb = embed(query).embedding
scores = [(cosine(q_emb, embed(s).embedding), s) for s in corpus]
scores.sort(reverse=True)

print(f"Query: '{query}'\n")
print(f"{'Rank':<5} {'Score':>6}  Sentence")
print("-" * 65)
for rank, (score, sent) in enumerate(scores, 1):
    print(f"{rank:<5} {score:>6.4f}  {sent}")

## 3. Batch Embedding and Pairwise Similarity Matrix

In [ ]:
sentences = [
    "İstanbul Türkiye'nin en kalabalık şehridir.",
    "Türkiye'de en çok insan İstanbul'da yaşar.",
    "Ankara başkent olarak idari merkez görevini üstlenmiştir.",
    "Türk mutfağı dünya genelinde çok beğenilmektedir.",
    "Türkiye'nin büyük şehirlerinden biri olan İzmir deniz kenarındadır.",
]

embeddings = [embed(s).embedding for s in sentences]

# Print the upper-triangle of the pairwise similarity matrix
header = "".join(f"  S{i+1}" for i in range(len(sentences)))
print(f"{'':>50}{header}")
for i, (s, e_i) in enumerate(zip(sentences, embeddings)):
    label = f"S{i+1}: {s[:42]:<42}"
    row   = "  " * i + "  --"
    for j in range(i + 1, len(sentences)):
        row += f" {cosine(e_i, embeddings[j]):>5.3f}"
    print(f"{label}  {row}")

## 4. Embeddings for Other Turkic Languages

The same `embeddings` processor is available for all languages supported by TurkicNLP. Simply change the language code.

In [ ]:
for lang, text in [
    ("kaz", "Бүгін ауа райы өте жақсы."),    # Kazakh
    ("kir", "Бүгүн аба ырайы абдан жакшы."),  # Kyrgyz
    ("uzb", "Bugun ob-havo juda yaxshi."),      # Uzbek
]:
    turkicnlp.download(lang, processors=["embeddings"])
    pipe = Pipeline(lang, processors=["embeddings"])
    emb  = pipe(text).embedding
    print(f"{lang}: dim={len(emb)},  norm={math.sqrt(sum(x**2 for x in emb)):.4f}")